# Billups Case Study — Answers

Each question below is a single cell that queries the **gold** Iceberg table
`billups.gold.transactions_enriched` (merchants ✕ historical transactions) using
the PySpark DataFrame API. The analytical functions live in
`billups_pipeline.transforms` so the notebook, the pipeline and the unit tests
all share identical logic.

Run the medallion build first (`make pipeline`, or trigger the Airflow DAG
`billups_medallion_pipeline`) so the gold table exists.

In [ ]:
from pyspark.sql import functions as F, Window

from billups_pipeline.config import T_GOLD_TRANSACTIONS
from billups_pipeline.session import build_spark
from billups_pipeline.transforms import (
    q1_top_merchants_by_city_month,
    q2_avg_sale_by_merchant_state,
    q3_top_hours_by_category,
    q4_popular_merchants_by_city,
)

spark = build_spark()
gold = spark.read.table(T_GOLD_TRANSACTIONS)


def show(df, n=20):
    """Render a Spark DataFrame as a pandas table for a clean notebook display."""
    return df.limit(n).toPandas()


print(f"gold rows: {gold.count():,}")
show(gold, 5)

## Question 1
Top 5 merchants by `purchase_amount` for each month, for each city.
`Purchase Total` = sum of purchase_amount; `No of sales` = transaction count.

In [ ]:
q1 = q1_top_merchants_by_city_month(gold, n=5)
# Example slice matching the spec layout (Oct/Nov 2017, a couple of cities).
show(
    q1.filter(F.col("month").isin("Oct 2017", "Nov 2017") & F.col("city").isin(2, 3)),
    40,
)

## Question 2
Average `purchase_amount` of each merchant in each state, largest sales first.

Note: the raw ranking is dominated by merchants with a single transaction
(average == that one sale). The second cell adds a `>= 500 transactions`
threshold for a more meaningful “high average-ticket” view.

In [ ]:
q2 = q2_avg_sale_by_merchant_state(gold)
display(show(q2, 20))

# Robustness variant: ignore tiny-sample merchants, then top merchant per state.
q2_robust = (
    gold.groupBy("merchant_name", "state_id")
    .agg(F.avg("purchase_amount").alias("average_amount"), F.count(F.lit(1)).alias("n"))
    .filter(F.col("n") >= 500)
)
w = Window.partitionBy("state_id").orderBy(F.desc("average_amount"))
q2_robust = (
    q2_robust.withColumn("rank_in_state", F.row_number().over(w))
    .filter(F.col("rank_in_state") == 1)
    .orderBy(F.desc("average_amount"))
)
show(q2_robust, 25)

## Question 3
Top 3 hours by total `purchase_amount` for each product category.
Hours are shown on a 24h clock (e.g. `1300`).

In [ ]:
q3 = q3_top_hours_by_category(gold, n=3)
show(q3, 100)

## Question 4
Most popular merchants (by number of sales transactions) and the city they sell
in, plus whether location (`city_id`) correlates with category (`category`).

In [ ]:
q4 = q4_popular_merchants_by_city(gold, n=20)
display(show(q4, 20))

# Correlation between city_id and category via Cramér's V on the contingency table.
import numpy as np

ct = gold.groupBy("city_id", "category").agg(F.count(F.lit(1)).alias("n")).toPandas()
piv = ct.pivot_table(index="city_id", columns="category", values="n", fill_value=0).values
N = piv.sum()
exp = piv.sum(1, keepdims=True) @ piv.sum(0, keepdims=True) / N
chi2 = ((piv - exp) ** 2 / exp).sum()
r, k = piv.shape
cramers_v = np.sqrt((chi2 / N) / min(r - 1, k - 1))
print(f"Cramér's V (city_id vs category) = {cramers_v:.3f}  (0=independent, 1=perfect)")

# Category mix within the busiest cities highlights the relationship.
top_cities = [r2[0] for r2 in gold.groupBy("city_id").count().orderBy(F.desc("count")).limit(6).collect()]
mix = (
    gold.filter(F.col("city_id").isin(top_cities))
    .groupBy("city_id", "category")
    .agg(F.count(F.lit(1)).alias("n"))
)
tot = gold.filter(F.col("city_id").isin(top_cities)).groupBy("city_id").agg(F.count(F.lit(1)).alias("t"))
show(
    mix.join(tot, "city_id")
    .withColumn("pct", F.round(100 * F.col("n") / F.col("t"), 1))
    .orderBy("city_id", "category"),
    40,
)

## Question 5 — Advice for a new merchant
Assumptions are stated in the README. Below we compute the evidence behind each
recommendation: (a) cities, (b) categories, (c) seasonality, (d) trading hours,
(e) the installments / credit-default profit model.

In [ ]:
g = gold  # alias

print("a) Top cities by revenue & volume")
display(show(
    g.groupBy("city_id").agg(
        F.sum("purchase_amount").alias("revenue"),
        F.count(F.lit(1)).alias("txns"),
        F.round(F.avg("purchase_amount"), 2).alias("avg_ticket"),
    ).orderBy(F.desc("revenue")),
    10,
))

print("b) Categories by revenue & volume")
display(show(
    g.groupBy("category").agg(
        F.sum("purchase_amount").alias("revenue"),
        F.count(F.lit(1)).alias("txns"),
        F.round(F.avg("purchase_amount"), 2).alias("avg_ticket"),
    ).orderBy(F.desc("revenue")),
    10,
))

print("c) Monthly seasonality")
display(show(
    g.groupBy("txn_month_key", "txn_month_label").agg(
        F.sum("purchase_amount").alias("revenue"),
        F.count(F.lit(1)).alias("txns"),
    ).orderBy("txn_month_key"),
    20,
))

print("d) Hour-of-day trading curve (hour 0 is inflated by date-only timestamps)")
display(show(
    g.groupBy("txn_hour").agg(
        F.sum("purchase_amount").alias("revenue"),
        F.count(F.lit(1)).alias("txns"),
    ).orderBy("txn_hour"),
    24,
))

In [ ]:
# e) Installments & credit default.
# Assumptions: gross profit = 25% of purchase_amount; defaulters pay exactly half
# the price; monthly default hazard 22.9% compounds over the installment horizon
# n, so P(default) = 1 - (1 - 0.229)**n. installments <= 1 => paid up front,
# no credit exposure.
#   received  = (1-p)*price + p*0.5*price = price*(1 - 0.5p)
#   cost      = 0.75*price       (since 25% gross margin)
#   profit    = price*(0.25 - 0.5p)
RATE = 0.229

def profit_factor(n):
    if n is None or n <= 1:
        return 0.25  # cash / single payment
    p = 1 - (1 - RATE) ** n
    return 0.25 - 0.5 * p

pf_udf = F.udf(lambda n: float(profit_factor(n)))

inst = (
    g.groupBy("installments")
    .agg(F.sum("purchase_amount").alias("revenue"), F.count(F.lit(1)).alias("txns"))
    .withColumn("profit_factor", pf_udf(F.col("installments")).cast("double"))
    .withColumn("expected_profit", F.round(F.col("profit_factor") * F.col("revenue"), 0))
    .withColumn("cash_profit", F.round(0.25 * F.col("revenue"), 0))
    .orderBy("installments")
)
show(inst, 40)